# Assignment 1: Model Repair

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/06/Assignment_01_Model_Repair.ipynb)

**Course:** Neural Architectures and Representation Learning  
**Related notebook:** `Week_06_Fixing_Models_Generalization_Stability.ipynb`

## Task

You are given a fixed dataset and a fixed broken baseline model. Your job is to repair the model and justify your choices using train/validation evidence.

The goal is not to try every trick. The goal is to diagnose the failure and apply a small set of appropriate fixes.

This assignment uses a different dataset from the teaching notebook: an 8-dimensional synthetic classification task with noisy distractor features.

## Submission requirements

Submit a completed copy of this notebook.

Your submission must include:

1. The provided baseline run.
2. At least **three repair experiments**.
3. A final repaired model.
4. Train/validation loss and accuracy plots.
5. Short written answers explaining your diagnosis and choices.

Do not change the fixed dataset cell or the baseline cell. This keeps submissions comparable.

Your coding work happens in:

- Experiment 1
- Experiment 2
- Experiment 3
- Final repaired model

Important: the starter experiment cells intentionally begin close to the broken baseline. Running the notebook without meaningful edits does not satisfy the assignment.

## Grading rubric

| Criterion | Points |
|----------|--------|
| Correctly runs the fixed baseline and reports its behavior | 20 |
| Performs controlled repair experiments | 25 |
| Uses appropriate repair tools and compares against baseline | 25 |
| Provides clear plots and metrics | 15 |
| Explains diagnosis, tradeoffs, and final choice | 15 |

Total: 100 points.

---

## Environment

This notebook uses `torch`, `numpy`, and `matplotlib`. CPU is enough.

In [ ]:
import copy
import numpy as np
import torch
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.rc("axes", grid=True)

%matplotlib inline

print("torch:", torch.__version__)
print("numpy:", np.__version__)

major = int(torch.__version__.split(".")[0])
assert major >= 2, "This notebook expects PyTorch 2.x or newer."

device = torch.device("cpu")

---

## 1. Fixed setup

Do not modify this section. Everyone uses the same 8D data, split, baseline architecture, and metrics.

In [ ]:
DATA_SEED = 109
TRAIN_SIZE = 240
VAL_SIZE = 800
N_FEATURES = 8
LABEL_NOISE = 0.35


def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_assignment_data(n_train=TRAIN_SIZE, n_val=VAL_SIZE, n_features=N_FEATURES, noise=LABEL_NOISE, seed=DATA_SEED):
    rng = np.random.default_rng(seed)

    def sample(n):
        X = rng.normal(0.0, 1.0, size=(n, n_features)).astype(np.float32)

        # Signal uses only the first four dimensions.
        # The remaining dimensions are nuisance features that can invite overfitting.
        signal = (
            1.15 * np.sin(1.4 * X[:, 0])
            + 0.85 * (X[:, 1] ** 2 - 0.8)
            - 0.75 * X[:, 2]
            + 0.55 * X[:, 0] * X[:, 3]
        )
        distractor = 0.20 * X[:, 4] - 0.15 * X[:, 5] + 0.10 * rng.normal(size=n)
        score = signal + distractor + rng.normal(0.0, noise, size=n)
        threshold = np.median(score)
        y = (score > threshold).astype(np.float32).reshape(-1, 1)
        return X, y

    X_train, y_train = sample(n_train)
    X_val, y_val = sample(n_val)

    return (
        torch.tensor(X_train, device=device),
        torch.tensor(y_train, device=device),
        torch.tensor(X_val, device=device),
        torch.tensor(y_val, device=device),
    )


X_train, y_train, X_val, y_val = make_assignment_data()

plt.figure(figsize=(6, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train.squeeze(), cmap="coolwarm", s=32, edgecolor="black", linewidth=0.4)
plt.xlabel("x0 (signal feature)")
plt.ylabel("x1 (signal feature)")
plt.title("Fixed assignment data: 2D view of an 8D problem")
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()

print("X_train shape:", tuple(X_train.shape))
print("X_val shape:  ", tuple(X_val.shape))

In [ ]:
class MLPClassifier(torch.nn.Module):
    def __init__(
        self,
        hidden_width=64,
        n_hidden_layers=3,
        activation="relu",
        dropout=0.0,
        batch_norm=False,
        init="kaiming",
        init_scale=1.0,
    ):
        super().__init__()
        activation = activation.lower()
        layers = []
        in_features = N_FEATURES

        for _ in range(n_hidden_layers):
            linear = torch.nn.Linear(in_features, hidden_width)
            layers.append(linear)
            if batch_norm:
                layers.append(torch.nn.BatchNorm1d(hidden_width))
            if activation == "relu":
                layers.append(torch.nn.ReLU())
            elif activation == "tanh":
                layers.append(torch.nn.Tanh())
            else:
                raise ValueError("activation must be 'relu' or 'tanh'")
            if dropout > 0:
                layers.append(torch.nn.Dropout(dropout))
            in_features = hidden_width

        layers.append(torch.nn.Linear(in_features, 1))
        self.net = torch.nn.Sequential(*layers)
        self.reset_parameters(init=init, activation=activation, init_scale=init_scale)

    def reset_parameters(self, init="kaiming", activation="relu", init_scale=1.0):
        for module in self.modules():
            if isinstance(module, torch.nn.Linear):
                if init == "kaiming":
                    torch.nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
                elif init == "xavier":
                    gain = torch.nn.init.calculate_gain("tanh" if activation == "tanh" else "relu")
                    torch.nn.init.xavier_normal_(module.weight, gain=gain)
                elif init == "normal":
                    torch.nn.init.normal_(module.weight, mean=0.0, std=init_scale)
                else:
                    raise ValueError("init must be 'kaiming', 'xavier', or 'normal'")
                torch.nn.init.zeros_(module.bias)

    def forward(self, X):
        return self.net(X)


def accuracy_from_logits(logits, y):
    preds = (torch.sigmoid(logits) >= 0.5).float()
    return float((preds == y).float().mean())


def train_model(
    config,
    lr=0.01,
    n_epochs=400,
    seed=42,
    weight_decay=0.0,
    grad_clip=None,
    early_stopping=False,
    patience_epochs=30,
):
    set_seed(seed)
    model = MLPClassifier(**config).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "grad_norm": []}
    best_state = copy.deepcopy(model.state_dict())
    best_val = float("inf")
    wait = 0

    for _ in range(n_epochs):
        model.train()
        loss = loss_fn(model(X_train), y_train)

        optimizer.zero_grad()
        loss.backward()

        total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip or 1e9)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            train_logits = model(X_train)
            val_logits = model(X_val)
            train_loss = loss_fn(train_logits, y_train)
            val_loss = loss_fn(val_logits, y_val)

            history["train_loss"].append(float(train_loss))
            history["val_loss"].append(float(val_loss))
            history["train_acc"].append(accuracy_from_logits(train_logits, y_train))
            history["val_acc"].append(accuracy_from_logits(val_logits, y_val))
            history["grad_norm"].append(float(total_norm))

        if not np.isfinite(history["train_loss"][-1]) or not np.isfinite(history["val_loss"][-1]):
            break

        if early_stopping:
            if history["val_loss"][-1] < best_val:
                best_val = history["val_loss"][-1]
                best_state = copy.deepcopy(model.state_dict())
                wait = 0
            else:
                wait += 1
                if wait >= patience_epochs:
                    break

    if early_stopping:
        model.load_state_dict(best_state)

    return model, history


def plot_history(histories, title):
    plt.figure(figsize=(8, 4))
    for label, history in histories:
        plt.plot(np.maximum(history["train_loss"], 1e-8), linestyle="-", label=f"{label} train")
        plt.plot(np.maximum(history["val_loss"], 1e-8), linestyle="--", label=f"{label} val")
    plt.xlabel("Epoch")
    plt.ylabel("BCE loss")
    plt.yscale("log")
    plt.title(title)
    plt.grid(True, which="both", alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.show()


def print_summary(label, history):
    gap = history["train_acc"][-1] - history["val_acc"][-1]
    print(f"{label:18s} epochs={len(history['train_loss']):3d} train_acc={history['train_acc'][-1]:.3f} val_acc={history['val_acc'][-1]:.3f} gap={gap:.3f}")

---

## 2. Fixed broken baseline

Do not modify this cell. Run it and use it as your comparison point.

In [ ]:
BASELINE_CONFIG = dict(
    hidden_width=128,
    n_hidden_layers=4,
    activation="relu",
    dropout=0.0,
    batch_norm=False,
    init="kaiming",
)

BASELINE_LR = 0.01
BASELINE_EPOCHS = 450
BASELINE_SEED = 11

baseline_model, baseline_history = train_model(
    BASELINE_CONFIG,
    lr=BASELINE_LR,
    n_epochs=BASELINE_EPOCHS,
    seed=BASELINE_SEED,
    grad_clip=None,
    early_stopping=False,
)

plot_history([("broken baseline", baseline_history)], "Fixed broken baseline")
print_summary("broken baseline", baseline_history)

### Baseline diagnosis

Write 2-3 sentences:

- What symptom do you see?
- Is this mainly underfitting, overfitting, instability, or a mix?
- Which evidence supports your diagnosis?

YOUR ANSWER:

...

---

## 3. Repair experiments

Run at least three controlled experiments. Change one or two things at a time, and explain why.

Allowed repair tools:

- dropout
- batch normalization
- gradient clipping
- early stopping
- learning rate
- hidden width / number of layers, within a reasonable range
- activation and initialization
- weight decay

The data has noisy distractor features. Capacity control, dropout, weight decay, and early stopping are especially relevant.

Suggested ranges:

- `hidden_width`: 16, 32, 64, 96, 128
- `n_hidden_layers`: 1, 2, 3, 4
- `dropout`: 0.0 to 0.5
- `lr`: 0.003 to 0.02
- `weight_decay`: 0.0, 1e-4, 1e-3, 1e-2
- `grad_clip`: None, 0.5, 1.0, 5.0

Keep the dataset and baseline cell fixed.

In [ ]:
repair_experiments = []


def run_repair(label, config, lr=0.01, n_epochs=500, seed=11, grad_clip=None, early_stopping=False, patience_epochs=35, weight_decay=0.0):
    model, history = train_model(
        config,
        lr=lr,
        n_epochs=n_epochs,
        seed=seed,
        grad_clip=grad_clip,
        early_stopping=early_stopping,
        patience_epochs=patience_epochs,
        weight_decay=weight_decay,
    )
    repair_experiments.append((label, model, history))
    print_summary(label, history)
    return model, history

### Experiment 1

Choose one repair idea and test it. For example: reduce capacity, add dropout, or add weight decay.
Explain the idea in the markdown cell below.

In [ ]:
# TODO: choose your first repair settings.
# Starter values are intentionally close to the broken baseline.
experiment_1_config = dict(
    hidden_width=128,      # TODO: try smaller or keep baseline
    n_hidden_layers=4,     # TODO: try fewer layers or keep baseline
    activation="relu",
    dropout=0.0,           # TODO: try e.g. 0.1, 0.25, 0.4
    batch_norm=False,      # TODO: try True/False
    init="kaiming",
)

exp1_model, exp1_history = run_repair(
    "experiment 1",
    experiment_1_config,
    lr=0.01,               # TODO: try e.g. 0.003, 0.006, 0.01
    n_epochs=450,
    grad_clip=None,        # TODO: try None, 1.0
    early_stopping=False,  # TODO: try True/False
    weight_decay=0.0,      # TODO: try 1e-4, 1e-3, 1e-2
)

Why did you choose this change?

YOUR ANSWER:

...

### Experiment 2

Choose a different repair idea from Experiment 1. Do not just repeat the same settings.

In [ ]:
# TODO: choose your second repair settings.
experiment_2_config = dict(
    hidden_width=128,      # TODO
    n_hidden_layers=4,     # TODO
    activation="relu",
    dropout=0.0,           # TODO
    batch_norm=False,      # TODO
    init="kaiming",
)

exp2_model, exp2_history = run_repair(
    "experiment 2",
    experiment_2_config,
    lr=0.01,               # TODO
    n_epochs=450,
    grad_clip=None,        # TODO
    early_stopping=False,  # TODO
    weight_decay=0.0,      # TODO
)

Why did you choose this change?

YOUR ANSWER:

...

### Experiment 3

Choose a third repair idea or combine the best parts of Experiments 1 and 2.

In [ ]:
# TODO: choose your third repair settings.
experiment_3_config = dict(
    hidden_width=128,      # TODO
    n_hidden_layers=4,     # TODO
    activation="relu",
    dropout=0.0,           # TODO
    batch_norm=False,      # TODO
    init="kaiming",
)

exp3_model, exp3_history = run_repair(
    "experiment 3",
    experiment_3_config,
    lr=0.01,               # TODO
    n_epochs=450,
    grad_clip=None,        # TODO
    early_stopping=False,  # TODO
    patience_epochs=35,
    weight_decay=0.0,      # TODO
)

Why did you choose this change?

YOUR ANSWER:

...

---

## 4. Compare experiments

Your final choice should be based on validation behavior, not only training accuracy.

In [ ]:
all_histories = [("baseline", baseline_history)] + [(label, history) for label, _, history in repair_experiments]
plot_history(all_histories, "Baseline vs repair experiments")

print_summary("baseline", baseline_history)
for label, _, history in repair_experiments:
    print_summary(label, history)

Which experiment is your best candidate, and why?

YOUR ANSWER:

...

---

## 5. Final repaired model

Copy your best configuration below. You may make one final small adjustment, but explain it.

In [ ]:
# TODO: copy your best settings here.
# The starter values below are the broken baseline; improve them.
final_config = dict(
    hidden_width=128,      # TODO
    n_hidden_layers=4,     # TODO
    activation="relu",
    dropout=0.0,           # TODO
    batch_norm=False,      # TODO
    init="kaiming",
)

final_model, final_history = train_model(
    final_config,
    lr=0.01,               # TODO
    n_epochs=450,
    seed=11,
    grad_clip=None,        # TODO
    early_stopping=False,  # TODO
    patience_epochs=35,
    weight_decay=0.0,      # TODO
)

plot_history(
    [
        ("baseline", baseline_history),
        ("final repair", final_history),
    ],
    "Final repaired model vs baseline",
)

print_summary("baseline", baseline_history)
print_summary("final repair", final_history)

---

## 6. Final reflection

Answer briefly.

1. What was wrong with the baseline?
2. Which repair helped most?
3. Did any repair improve validation while reducing training performance?
4. What tradeoff did you accept?
5. What would you try next if you had more time?

YOUR ANSWER:

...

---

## Academic integrity and AI use

You may use AI tools for debugging and explanation support. You remain responsible for understanding and explaining your submitted notebook. Oral follow-up may be used to verify understanding.